# GeoSat 10-City Batch Processing (Colab Cloud GPU)

This notebook runs the land-cover change-detection pipeline for 10 major Indian cities sequentially, syncing results securely to Google Drive.

### Prerequisites:
1. **Enable GPU:** Go to `Runtime > Change runtime type` and select `T4 GPU` (or better).
2. **Google Drive:** You must have your trained model and your `service_account.json` saved in your Google Drive.

In [ ]:
# 1. Mount Google Drive & Check GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
print("\nCUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. Enable a GPU runtime in Google Colab (Runtime > Change runtime type).")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Configuration
# ---------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/LandCoverChangeResults"
MODEL_PATH = f"{DRIVE_ROOT}/models/resnet50_best.pt"
DRIVE_GEE_CREDENTIALS = f"{DRIVE_ROOT}/service_account.json"

START_YEAR = 2019
END_YEAR = 2023
BATCH_SIZE = 32
FORCE_RERUN = False

# PROCESS_MODE options: 'selected', 'all', 'failed', 'pending'
PROCESS_MODE = "selected"

# Active if PROCESS_MODE == 'selected'
SELECTED_CITIES = [
    "TinyMumbai"
]
# ---------------------------------------------------------

In [ ]:
# 3. Setup Environment & Clone Repository
import os
import shutil

!pip install -q earthengine-api torch torchvision rasterio pandas python-dotenv pyproj

if not os.path.exists('/content/Geo_Sat'):
    !git clone https://github.com/saran318/Geo_Sat.git /content/Geo_Sat

os.chdir('/content/Geo_Sat/geo_sat project')

os.makedirs('models', exist_ok=True)
if os.path.exists(MODEL_PATH):
    shutil.copy(MODEL_PATH, 'models/resnet50_best.pt')
    print("Model copied successfully.")
else:
    raise FileNotFoundError(f"Trained model not found at {MODEL_PATH}")

if os.path.exists(DRIVE_GEE_CREDENTIALS):
    shutil.copy(DRIVE_GEE_CREDENTIALS, 'service_account.json')
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.path.abspath('service_account.json')
    print("GEE credentials configured.")
else:
    raise FileNotFoundError(f"GEE credentials not found at {DRIVE_GEE_CREDENTIALS}")

In [ ]:
# 4. Define City Processing Logic
import json
import time
import datetime
import gc
import subprocess

# Load cities config
with open('config/cities.json', 'r') as f:
    ALL_CITIES_CONFIG = json.load(f)

os.makedirs(DRIVE_ROOT, exist_ok=True)
status_file = f"{DRIVE_ROOT}/processing_status.json"

if os.path.exists(status_file):
    with open(status_file, 'r') as f:
        processing_status = json.load(f)
else:
    processing_status = {}

# Determine target cities based on PROCESS_MODE
target_cities = []
if PROCESS_MODE == 'selected':
    target_cities = SELECTED_CITIES
elif PROCESS_MODE == 'all':
    target_cities = [c for c in ALL_CITIES_CONFIG.keys() if c != 'TinyMumbai']
elif PROCESS_MODE == 'failed':
    target_cities = [c for c, stat in processing_status.items() if stat.get('status') == 'failed']
elif PROCESS_MODE == 'pending':
    target_cities = [c for c in ALL_CITIES_CONFIG.keys() if c != 'TinyMumbai' and processing_status.get(c, {}).get('status') != 'completed']

print(f"Target cities to process ({len(target_cities)}): {target_cities}")

In [ ]:
# 5. Execute Pipeline Sequentially
for city in target_cities:
    print(f"\n{'='*50}")
    print(f"Processing City: {city}")
    print(f"{'='*50}")
    
    drive_city_dir = f"{DRIVE_ROOT}/{city}"
    drive_completed = f"{drive_city_dir}/completed.json"
    
    if not FORCE_RERUN and os.path.exists(drive_completed):
        print(f"City {city} already completed. Skipping.")
        continue
        
    processing_status[city] = {
        "status": "running",
        "start_time": datetime.datetime.now().isoformat()
    }
    with open(status_file, 'w') as f:
        json.dump(processing_status, f, indent=4)
        
    start_time = time.time()
    try:
        # Run the pipeline script
        cmd = f"python run_real_pipeline.py --city {city} --years {START_YEAR} {END_YEAR} --batch_size {BATCH_SIZE}"
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        
        if result.returncode != 0:
            raise RuntimeError(f"Pipeline script failed: {result.stderr}")
            
        duration = time.time() - start_time
        
        # Sync to Drive
        local_results = f"data/results/{city}"
        if os.path.exists(local_results):
            os.makedirs(drive_city_dir, exist_ok=True)
            # Copy directories recursively
            for item in os.listdir(local_results):
                s = os.path.join(local_results, item)
                d = os.path.join(drive_city_dir, item)
                if os.path.isdir(s):
                    if os.path.exists(d): shutil.rmtree(d)
                    shutil.copytree(s, d)
                else:
                    shutil.copy2(s, d)
            
            processing_status[city].update({
                "status": "completed",
                "end_time": datetime.datetime.now().isoformat(),
                "duration_sec": round(duration, 2),
                "output_path": drive_city_dir
            })
            print(f"Successfully processed and synced {city} in {duration:.2f}s")
        else:
            raise FileNotFoundError(f"Local results not found at {local_results}")
            
    except Exception as e:
        print(f"ERROR processing {city}: {e}")
        processing_status[city].update({
            "status": "failed",
            "end_time": datetime.datetime.now().isoformat(),
            "error": str(e)
        })
        
    # Save status
    with open(status_file, 'w') as f:
        json.dump(processing_status, f, indent=4)
        
    # Cleanup and release GPU memory before next city
    torch.cuda.empty_cache()
    gc.collect()
    if os.path.exists(f"data/results/{city}"):
        shutil.rmtree(f"data/results/{city}")

In [ ]:
# 6. Final Summary
print("\n" + "*"*60)
print("FINAL PROCESSING SUMMARY")
print("*"*60)
for c, stat in processing_status.items():
    status = stat.get('status', 'unknown')
    dur = stat.get('duration_sec', 'N/A')
    err = stat.get('error', '')
    print(f"{c:15s} | {status:10s} | {dur}s | {err}")
print("*"*60)